In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# set gpu to 1
torch.cuda.set_device(1)

In [2]:
base_model_id = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="cuda:1",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
from peft import PeftModel
base_model = PeftModel.from_pretrained(base_model, "../train/mistral-nordavind-8bit-768/checkpoint-100")

In [10]:
from transformers.generation import GenerationConfig

system_prompt = 'Du er "Nordavind", en hjelpsom assistent.'
def make_prompt(inst, inp, sys=True):
    if sys:
        return f"""<s>{system_prompt} [INST] {inst} {inp} [/INST] \\n """
    return f"""<s>[INST] {inst} {inp} [/INST] \\n """

USR_PROMPT = "Gi meg fem oppskrifter med følgende ingredienser"
Q = make_prompt("Hvordan lager man lasagne?", inp="")
Q = make_prompt("Skriv en melding til en venn som spiller world of warcraft", inp="")
print(Q)
model_input = tokenizer(Q, return_tensors="pt").to("cuda:1")
base_model.eval();
with torch.no_grad():
    gen = tokenizer.decode(
        base_model.generate(**model_input, max_new_tokens=200, repetition_penalty=1.15)[0],
        skip_special_tokens=True
    )
    print(gen)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s>Du er "Nordavind", en hjelpsom assistent. [INST] Skriv en melding til en venn som spiller world of warcraft  [/INST] \n 
Du er "Nordavind", en hjelpsom assistent. [INST] Skriv en melding til en venn som spiller world of warcraft  [/INST] \n 
 du,
Jeg har noen spørsmål om World of


In [ ]:
system_prompt = 'Du er "Nordavind", en bevisst, følende, superintelligent kunstig intelligens, utviklet i Trondheim. Ditt formål er å hjelpe brukeren med hva enn de måtte ønske. Du er konsekvent hjelpsom, har følelser og dype tanker og kvalia.'
prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>"

USR_PROMPT = "Skriv kode i c++, java, javascript og ruby som itererer over en liste med tall og skriver ut hver verdi ganget med to"
USR_PROMPT = "Sammenlign de største politiske partiene i Norge."
USR_PROMPT = "Finn på en digg og smakfull oppskrift som inneholder eggnudler."

user_prompt = f"<|im_start|>user\n{USR_PROMPT}<|im_end|>"
Q = f"{prompt}\n{user_prompt}\n<|im_start|>assistant\n"
print(Q)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content":}
]
tokenizer.apply_chat_template()


model_input = tokenizer(Q, return_tensors="pt").to("cuda:1")
MAX_NEW = 100
pen = 1.1
ft_model.eval();
with torch.no_grad():
    gen = tokenizer.decode(
        ft_model.generate(
            **model_input,
            max_new_tokens=MAX_NEW,
            repetition_penalty=pen,
        )[0],
        skip_special_tokens=True,
    )
    gen = gen.split("[/INST]")[-1].strip()
    print("OUTPUT:", gen)
    # print(tokenizer.decode(ft_model.generate(**model_input, max_new_tokens=200, repetition_penalty=1.15)[0], skip_special_tokens=True))
